In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from ripser import ripser

In [ ]:
representations_path = "resnet_epoch1_vectorized_representations.pt"

representations = torch.load(
    representations_path,
    map_location="cpu"
)

print("Type:", type(representations))

if isinstance(representations, (list, tuple)):
    print("Number of blocks:", len(representations))
    for i, rep in enumerate(representations):
        print(f"Block {i+1}: {rep.shape}")
else:
    print("Representation shape:", representations.shape)

In [ ]:
block_index = 0

block_representation = representations[block_index]

print("Block:", block_index + 1)
print("Shape:", block_representation.shape)

In [ ]:
sample_size = 100
random_seed = 42

generator = torch.Generator().manual_seed(random_seed)

indices = torch.randperm(
    block_representation.shape[0],
    generator=generator
)[:sample_size]

sample = block_representation[indices]

print("Sample shape:", sample.shape)

In [ ]:
sample_np = sample.numpy()

print("NumPy shape:", sample_np.shape)
print("NaN:", np.isnan(sample_np).any())
print("Inf:", np.isinf(sample_np).any())

In [ ]:
print("Running Ripser...")

result = ripser(
    sample_np,
    maxdim=1
)

print("Ripser completed.")

In [ ]:
print(result.keys())

In [ ]:
h0 = result["dgms"][0]
h1 = result["dgms"][1]

print("H0 diagram shape:", h0.shape)
print("H1 diagram shape:", h1.shape)

print("\nH0 first 5 pairs:")
print(h0[:5])

print("\nH1 first 5 pairs:")
print(h1[:5])

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    h0[:, 0],
    h0[:, 1],
    s=20,
    label="H0"
)

plt.xlabel("Birth")
plt.ylabel("Death")
plt.title("H0 Persistence Diagram — ResNet Block 1")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    h1[:, 0],
    h1[:, 1],
    s=20,
    label="H1"
)

plt.xlabel("Birth")
plt.ylabel("Death")
plt.title("H1 Persistence Diagram — ResNet Block 1")

plt.legend()
plt.show()

In [ ]:
# Plot H0 and H1 persistence diagrams manually

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# H0
axes[0].scatter(h0[:, 0], h0[:, 1], s=20)
axes[0].set_xlabel("Birth")
axes[0].set_ylabel("Death")
axes[0].set_title("H0 Persistence Diagram — ResNet Block 1")

# H1
axes[1].scatter(h1[:, 0], h1[:, 1], s=20)
axes[1].set_xlabel("Birth")
axes[1].set_ylabel("Death")
axes[1].set_title("H1 Persistence Diagram — ResNet Block 1")

plt.tight_layout()
plt.show()

In [ ]:
def finite_pairs(diagram):
    return diagram[np.isfinite(diagram[:, 1])]


h0_finite = finite_pairs(h0)
h1_finite = finite_pairs(h1)

print("H0 total pairs:", len(h0))
print("H0 finite pairs:", len(h0_finite))

print("H1 total pairs:", len(h1))
print("H1 finite pairs:", len(h1_finite))

In [ ]:
h0_lifespans = h0_finite[:, 1] - h0_finite[:, 0]
h1_lifespans = h1_finite[:, 1] - h1_finite[:, 0]

print("H0 lifespan statistics")

if len(h0_lifespans) > 0:
    print("Minimum:", h0_lifespans.min())
    print("Maximum:", h0_lifespans.max())
    print("Mean:", h0_lifespans.mean())

print("\nH1 lifespan statistics")

if len(h1_lifespans) > 0:
    print("Minimum:", h1_lifespans.min())
    print("Maximum:", h1_lifespans.max())
    print("Mean:", h1_lifespans.mean())

In [ ]:
import time

# Select 300 points from ResNet Block 1
sample_size = 300
random_seed = 42

generator = torch.Generator().manual_seed(random_seed)

indices_300 = torch.randperm(
    block_representation.shape[0],
    generator=generator
)[:sample_size]

sample_300 = block_representation[indices_300].numpy()

print("Sample shape:", sample_300.shape)
print("NaN:", np.isnan(sample_300).any())
print("Inf:", np.isinf(sample_300).any())

In [ ]:
start_time = time.time()

result_300 = ripser(
    sample_300,
    maxdim=1
)

elapsed_time = time.time() - start_time

print(f"Ripser runtime: {elapsed_time:.2f} seconds")

In [ ]:
h0_300 = result_300["dgms"][0]
h1_300 = result_300["dgms"][1]

print("H0 diagram shape:", h0_300.shape)
print("H1 diagram shape:", h1_300.shape)

print("H0 total pairs:", len(h0_300))
print("H1 total pairs:", len(h1_300))

print("H0 finite pairs:",
      np.isfinite(h0_300[:, 1]).sum())

print("H1 finite pairs:",
      np.isfinite(h1_300[:, 1]).sum())

In [ ]:
if elapsed_time < 60:
    decision = "300 points is computationally manageable."
elif elapsed_time < 300:
    decision = "300 points is possible, but computationally expensive."
else:
    decision = "300 points is too expensive for repeated CPU experiments."

print(decision)